In [1]:
# ============================================================
# INSTALL
# ============================================================
!pip install transformers datasets accelerate -q

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorWithPadding,
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 256
EPOCHS = 1

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
# ============================================================
# COMMON UTILITIES (USED BY PPO + DPO)
# ============================================================

def tokenize_pair(tokenizer, prompt, response):
    # return input_ids, attention_mask tensors and the prompt length in tokens
    p_enc = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)
    full_enc = tokenizer(prompt + response, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    input_ids = full_enc.input_ids.squeeze(0)
    attention_mask = full_enc.attention_mask.squeeze(0)
    prompt_len = len(p_enc["input_ids"])  # may be truncated
    return input_ids, attention_mask, prompt_len


def compute_logprob(model, input_ids, attention_mask, prompt_len=None):
    # compute log-probability of the sequence tokens predicted by the model
    # only sum logprobs for tokens that belong to the response (if prompt_len provided)
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
        attention_mask = attention_mask.unsqueeze(0)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # (batch, seq_len, vocab)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # next-token log-probs: prediction at position i predicts token at i+1
    shift_log_probs = log_probs[:, :-1, :]
    labels = input_ids[:, 1:]
    attn_labels = attention_mask[:, 1:]

    # build mask for which label positions to score
    if prompt_len is not None:
        # label index j corresponds to token position j+1 in the full sequence
        start_index = max(0, prompt_len - 1)
        score_positions = torch.arange(labels.size(1), device=labels.device).unsqueeze(0) >= start_index
        score_mask = score_positions & (attn_labels.bool())
    else:
        score_mask = attn_labels.bool()

    token_logps = shift_log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    token_logps = token_logps * score_mask.float()

    # sum across tokens -> per-example logprob
    summed = token_logps.sum(dim=1)
    return summed.squeeze()


def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    return output, prompt_len


In [4]:
# ============================================================
# DATASET
# ============================================================

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_pref = dataset["train"].select(range(2000))
val_pref = dataset["test"].select(range(500))

In [5]:
# ============================================================
# TOKENIZER
# ============================================================

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_size = "left"

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
# ============================================================
# ======================= SFT MODEL ==========================
# ============================================================

sft_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def sft_tokenize(example):
    # tokenize without forcing max-length padding — let the DataCollator handle padding
    text = example["prompt"] + example["chosen"]
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = enc["input_ids"].copy()
    return enc

sft_train = train_pref.map(sft_tokenize, remove_columns=["prompt","chosen","rejected"])
sft_val = val_pref.map(sft_tokenize, remove_columns=["prompt","chosen","rejected"])

# use DataCollatorWithPadding to dynamically pad batches
sft_collator = DataCollatorWithPadding(tokenizer)

train_loader = DataLoader(sft_train, batch_size=16, shuffle=True, collate_fn=sft_collator)
val_loader = DataLoader(sft_val, batch_size=16, collate_fn=sft_collator)

optimizer = torch.optim.AdamW(sft_model.parameters(), lr=2e-5)

for epoch in range(EPOCHS):
    sft_model.train()
    total = 0.0
    count = 0
    for batch in train_loader:
        # collator returns tensors — move them to device
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = sft_model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        count += 1
    print("SFT Epoch", epoch, total / max(1, count))

os.makedirs("anthropic_sft_model", exist_ok=True)
sft_model.save_pretrained("anthropic_sft_model")
tokenizer.save_pretrained("anthropic_sft_model")


/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is d

ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`labels` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:
# ============================================================
# ===================== REWARD MODEL =========================
# ============================================================

class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = nn.Linear(base.config.n_embd, 1)

    def forward(self, input_ids, attention_mask):
        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden = out.hidden_states[-1]
        return self.head(hidden).mean()

base = AutoModelForCausalLM.from_pretrained("anthropic_sft_model")
reward_model = RewardModel(base).to(device)

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    total = 0
    for ex in train_pref:
        chosen_ids, chosen_mask = tokenize_pair(tokenizer, ex["prompt"], ex["chosen"])
        rejected_ids, rejected_mask = tokenize_pair(tokenizer, ex["prompt"], ex["rejected"])

        chosen_ids = chosen_ids.unsqueeze(0).to(device)
        chosen_mask = chosen_mask.unsqueeze(0).to(device)
        rejected_ids = rejected_ids.unsqueeze(0).to(device)
        rejected_mask = rejected_mask.unsqueeze(0).to(device)

        c = reward_model(chosen_ids, chosen_mask)
        r = reward_model(rejected_ids, rejected_mask)

        loss = loss_fn(c - r, torch.ones_like(c))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    print("Reward Epoch", epoch, total/len(train_pref))

torch.save(reward_model.state_dict(), "anthropic_reward.pt")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5412.43it/s]


Reward Epoch 0 0.0024930380346717116


In [ ]:
ppo_policy = AutoModelForCausalLM.from_pretrained("anthropic_sft_model").to(device)
ppo_ref = AutoModelForCausalLM.from_pretrained("anthropic_sft_model").to(device)

for p in ppo_ref.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=1e-5)
beta = 0.02
clip_eps = 0.2

for epoch in range(EPOCHS):
    total = 0
    for ex in train_pref:

        generated = generate_response(ppo_policy, tokenizer, ex["prompt"])
        mask = torch.ones_like(generated).to(device)

        reward = reward_model(generated, mask)

        logp = compute_logprob(ppo_policy, generated, mask)
        logp_ref = compute_logprob(ppo_ref, generated, mask)

        ratio = torch.exp(logp - logp_ref)
        clipped = torch.clamp(ratio, 1-clip_eps, 1+clip_eps)

        loss = -torch.min(ratio*reward, clipped*reward) + beta*(logp-logp_ref)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    print("PPO Epoch", epoch, total/len(train_pref))

ppo_policy.save_pretrained("ppo_anthropic_model")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10481.16it/s]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end genera

KeyboardInterrupt: 

In [ ]:
# ============================================================
# ======================== DPO MODEL =========================
# ============================================================

dpo_policy = AutoModelForCausalLM.from_pretrained("anthropic_sft_model").to(device)
dpo_ref = AutoModelForCausalLM.from_pretrained("anthropic_sft_model").to(device)

for p in dpo_ref.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(dpo_policy.parameters(), lr=5e-5)
beta = 0.1
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    total = 0
    for ex in train_pref:

        c_ids, c_mask = tokenize_pair(tokenizer, ex["prompt"], ex["chosen"])
        r_ids, r_mask = tokenize_pair(tokenizer, ex["prompt"], ex["rejected"])

        c_ids = c_ids.unsqueeze(0).to(device)
        r_ids = r_ids.unsqueeze(0).to(device)
        c_mask = c_mask.unsqueeze(0).to(device)
        r_mask = r_mask.unsqueeze(0).to(device)

        logp_c = compute_logprob(dpo_policy, c_ids, c_mask)
        logp_r = compute_logprob(dpo_policy, r_ids, r_mask)

        logp_c_ref = compute_logprob(dpo_ref, c_ids, c_mask)
        logp_r_ref = compute_logprob(dpo_ref, r_ids, r_mask)

        logits = beta * ((logp_c - logp_c_ref) - (logp_r - logp_r_ref))
        loss = loss_fn(logits, torch.ones_like(logits))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    print("DPO Epoch", epoch, total/len(train_pref))

dpo_policy.save_pretrained("dpo_anthropic_model")

print("\nFINAL MODELS SAVED:")
print("PPO → ppo_anthropic_model")
print("DPO → dpo_anthropic_model")